# 03 - Stratified Train / Validation / Test Split

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path(os.getcwd())
for _root in [_cwd, *_cwd.parents]:
    if (_root / "skin_lesion" / "src" / "config.py").exists():
        sys.path.insert(0, str(_root / "skin_lesion" / "src"))
        break

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from config import SEED, PROCESSED_DIR, TABLES_DIR

## 1 - Load features

In [ ]:
data = np.load(PROCESSED_DIR / "features_hsv.npz", allow_pickle=True)
X, y = data["X"], data["y"]
image_ids = data["image_ids"].astype(str)

subset = pd.read_csv(PROCESSED_DIR / "balanced_subset.csv")
subset["image_id"] = subset["image_id"].astype(str)
id2lesion = dict(zip(subset["image_id"], subset["lesion_id"]))
groups = np.array([id2lesion[i] for i in image_ids])

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Class counts — melanoma: {(y==1).sum()}, benign: {(y==0).sum()}")
print(f"Unique lesions (groups): {len(np.unique(groups))}")

X shape : (2226, 96)
y shape : (2226,)
Class counts — melanoma: 1113, benign: 1113
Unique lesions (groups): 1695


## 2 - Group-aware stratified split

Several images can belong to the same physical lesion (lesion_id). 
`StratifiedGroupKFold` keeps every lesion inside one split while preserving balance to avoid leakage.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = [test_idx for _, test_idx in sgkf.split(X, y, groups)]

test_idx  = folds[0]                    
val_idx   = folds[1]                    
train_idx = np.concatenate(folds[2:])   

X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

# Sanity: no lesion_id may appear in more than one split.
g_train, g_val, g_test = set(groups[train_idx]), set(groups[val_idx]), set(groups[test_idx])
assert not (g_train & g_val) and not (g_train & g_test) and not (g_val & g_test), \
    "Lesion-id leakage detected between splits!"

print(f"Train : {X_train.shape}  melanoma={y_train.sum()}  benign={(y_train==0).sum()}")
print(f"Val   : {X_val.shape}    melanoma={y_val.sum()}  benign={(y_val==0).sum()}")
print(f"Test  : {X_test.shape}   melanoma={y_test.sum()}  benign={(y_test==0).sum()}")
print(f"Total : {len(y_train)+len(y_val)+len(y_test)}  (original: {len(y)})")
print(f"Unique lesions - train={len(g_train)}, val={len(g_val)}, test={len(g_test)}  (0 shared)")

Train : (1335, 96)  melanoma=667  benign=668
Val   : (445, 96)    melanoma=223  benign=222
Test  : (446, 96)   melanoma=223  benign=223
Total : 2226  (original: 2226)
Unique lesions - train=1017, val=338, test=340  (0 shared)


## 3 - Confirm stratification

The melanoma fraction should be ≈ 0.50 in every split.

In [4]:
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    frac = yy.mean()
    print(f"  {name:5s}: melanoma fraction = {frac:.4f}")

  train: melanoma fraction = 0.4996
  val  : melanoma fraction = 0.5011
  test : melanoma fraction = 0.5000


## 4 - Save splits

In [7]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / "splits.npz"
np.savez_compressed(
    out_path,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test,
)
print(f"Splits saved to: {out_path}")

Splits saved to: C:\Users\minef\Downloads\Skin\skin_lesion_triage\skin_lesion\data\processed\splits.npz


## 5 - Summary 

In [6]:
rows = []
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    rows.append({
        "split":      name,
        "n_total":    len(yy),
        "n_melanoma": int(yy.sum()),
        "n_benign":   int((yy == 0).sum()),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

TABLES_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(TABLES_DIR / "split_summary.csv", index=False)
print(f"\nSaved to: {TABLES_DIR / 'split_summary.csv'}")

split  n_total  n_melanoma  n_benign
train     1335         667       668
  val      445         223       222
 test      446         223       223

Saved to: C:\Users\minef\Downloads\Skin\skin_lesion_triage\skin_lesion\results\tables\split_summary.csv
